# Artifact parser test

Interactive checks for `shared.artifacts.ArtifactHandler`.

Parse returns **addressable blocks**: agents cite `block_id`; a viewer can open `page` and highlight `bbox`.

Supported files: `.pdf`, `.pptx`, `.docx`, `.xlsx` / `.xlsm`.

1. Set `FILE_PATH` in the config cell.
2. Run **classify**, then **parse**.
3. Inspect `to_dict()` (`artifact_id`, `pages`, `blocks`, `markdown`) and loc comments.

In [22]:
from __future__ import annotations

import json
import os
import sys
from collections import Counter
from pathlib import Path

from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
repo_root = next(
    (p for p in [cwd, *cwd.parents] if (p / "shared" / "artifacts").is_dir()),
    cwd,
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from shared.artifacts import ArtifactHandler, ParseOptions, citation_from_block
from shared.artifacts.exceptions import ArtifactError

print(f"repo_root = {repo_root}")

repo_root = D:\Work\Etex\ai_application


## Config

Point `FILE_PATH` at a local pdf / pptx / docx / xlsx. Leave `ARTIFACT_TYPE` as `None` to classify first.

PDF image/chart blocks are emitted structurally (not gated on `include_images`). Set `use_ocr=True` only for scanned pages.

In [ ]:
FIXTURES = Path(os.environ.get("ARTIFACT_FIXTURES_DIR", r"D:\Work\Etex\Simple example"))
FILE_PATH = FIXTURES / "KPMG_ADVISORY_SRL_BE-X-439819279_2026-03-18_14-39-57.pdf"
ARTIFACT_TYPE = None  # or "pdf" | "ppt" | "excel" | "word"
PARSE_OPTIONS = ParseOptions(
    password=None,
    include_tables=True,
    include_images=False,
    include_hidden_sheets=False,
    max_pages=None,
    use_ocr=False,
)

SAVE_JSON = repo_root / "test_files" / "parser_test" / "sample_output.json"
SAVE_MD = repo_root / "test_files" / "parser_test" / f"{FILE_PATH.stem}.md"

path = FILE_PATH.expanduser().resolve()
print(f"exists={path.is_file()}  path={path}")
print(f"save json={SAVE_JSON}")
print(f"save md={SAVE_MD}")

exists=True  path=D:\Work\Etex\Simple example\Etex_RFP_Swift CSP_06022026.pdf
save json=D:\Work\Etex\ai_application\test_files\parser_test\sample_output.json
save md=D:\Work\Etex\ai_application\test_files\parser_test\Etex_RFP_Swift CSP_06022026.md


## Classify

In [24]:
handler = ArtifactHandler()
kind = handler.classify(path)
print(json.dumps({"path": str(path), "action": "classify", "artifact_type": kind.value}, indent=2))

{
  "path": "D:\\Work\\Etex\\Simple example\\Etex_RFP_Swift CSP_06022026.pdf",
  "action": "classify",
  "artifact_type": "pdf"
}


## Parse

`document.to_dict()` is the JSON contract: `artifact_id`, `coord_system`, `pages`, `outline`, `blocks`, `warnings`, `markdown`. No `action` / `plain_text` / `MD_text` keys.

In [25]:
try:
    document = handler.parse(path, ARTIFACT_TYPE, options=PARSE_OPTIONS)
except ArtifactError as exc:
    raise SystemExit(f"parse failed: {exc}") from exc

payload = document.to_dict()
print(
    f"type={document.artifact_type.value}  artifact_id={document.artifact_id}  "
    f"coord={document.coord_system}  pages={len(document.pages)}  "
    f"blocks={len(document.blocks)}  outline={len(document.outline)}"
)
print("warnings:", document.warnings or "none")
print("to_dict keys:", list(payload))
print(json.dumps(document.metadata.to_dict(), indent=2, default=str))

type=pdf  artifact_id=sha256:6797dc2b89f92cde  coord=pdf_points_top_left  pages=15  blocks=179  outline=1
warnings: none
to_dict keys: ['artifact_id', 'source', 'artifact_type', 'coord_system', 'metadata', 'pages', 'outline', 'blocks', 'warnings', 'markdown']
{
  "filename": "Etex_RFP_Swift CSP_06022026.pdf",
  "artifact_type": "pdf",
  "mime_type": "application/pdf",
  "title": null,
  "author": "Student EHS",
  "created": "D:20260206101457+01'00'",
  "modified": "D:20260206101457+01'00'",
  "page_count": 15,
  "slide_count": null,
  "sheet_count": null,
  "encrypted": false
}


## Block summary

`id` is a debug sequence. Agents cite **`block_id`**. `heading_path` is the section stack at that block.

In [26]:
print(dict(Counter(block.type.value for block in document.blocks)))
print("unique block_id:", len({block.block_id for block in document.blocks}) == len(document.blocks))
print()
for block in document.blocks[:20]:
    loc = block.location.to_dict()
    where = {k: loc[k] for k in ("page", "slide", "sheet", "bbox") if k in loc}
    preview = block.text.replace("\n", " ")[:90]
    heading = " > ".join(block.heading_path[-2:]) if block.heading_path else ""
    print(f"{block.id:12} {block.block_id:16} {block.type.value:8} {where}  {heading}  {preview}")

{'table': 17, 'text': 145, 'image': 5, 'list': 11, 'heading': 1}
unique block_id: True

pdf-0001     b_ff77d37f7efb   table    {'page': 1, 'bbox': [14.9, 8.8, 582.8, 82.0]}    Request for Proposal |  Etex – SWIFT CSP assessment | Version: 1 Date: 27 January 2026 | P
pdf-0003     b_f91d46179d47   text     {'page': 1, 'bbox': [247.1, 256.3, 351.2, 267.5]}    SWIFT CSP assessment
pdf-0002     b_8ca757279606   image    {'page': 1, 'bbox': [1.8, 365.5, 331.8, 846.6]}    [image]
pdf-0004     b_832da68ff19d   table    {'page': 2, 'bbox': [14.9, 8.8, 582.8, 82.0]}    Request for Proposal |  Etex – SWIFT CSP assessment | Version: 1 Date: 27 January 2026 | P
pdf-0005     b_0778de4e380e   text     {'page': 2, 'bbox': [72.0, 103.1, 169.8, 115.0]}    TABLE OF CONTENTS
pdf-0006     b_94a202586aa7   text     {'page': 2, 'bbox': [83.2, 135.9, 525.7, 166.1]}    Introduction..............................................................................
pdf-0007     b_b840730b09d6   text     {'page': 2, '

## Citation helper

Not parser output. `citation_from_block(...)` is what an agent can return so a viewer can open the box.

In [27]:
fee = next(
    (
        block
        for block in document.blocks
        if block.type.value == "table" and "35.251" in (block.text or "")
    ),
    None,
)
target = fee or next((block for block in document.tables()), None)
if target is None:
    target = document.blocks[0]

citation = citation_from_block(target, document.artifact_id, document.coord_system)
print(json.dumps(citation, indent=2, ensure_ascii=False))
print("heading_path:", target.heading_path)

{
  "block_id": "b_ff77d37f7efb",
  "artifact_id": "sha256:6797dc2b89f92cde",
  "type": "table",
  "page": 1,
  "bbox": [
    14.9,
    8.8,
    582.8,
    82.0
  ],
  "page_width": 595.4,
  "page_height": 842.0,
  "coord_system": "pdf_points_top_left",
  "heading_path": [],
  "quote": "Request for Proposal |"
}
heading_path: []


## Save Markdown

Writes `document.md_text()` to a `.md` file next to this notebook. Location is an HTML comment only: `<!-- loc page=N block=b_xxx type=table -->`.

In [28]:
markdown = document.md_text()
if markdown and not markdown.endswith("\n"):
    markdown += "\n"
SAVE_MD.parent.mkdir(parents=True, exist_ok=True)
SAVE_MD.write_text(markdown, encoding="utf-8")
print(f"wrote {SAVE_MD}  ({SAVE_MD.stat().st_size:,} bytes)")
display(Markdown(markdown[:4000] + ("\n\n…" if len(markdown) > 4000 else "")))

wrote D:\Work\Etex\ai_application\test_files\parser_test\Etex_RFP_Swift CSP_06022026.md  (40,315 bytes)


---
artifact_id: "sha256:6797dc2b89f92cde"
filename: Etex_RFP_Swift CSP_06022026.pdf
coord_system: pdf_points_top_left
pages: 15
---

## Page 1

| Request for Proposal |  |
| --- | --- |
| Etex – SWIFT CSP assessment | Version: 1 |
| Date: 27 January 2026 | Page: 1 of 15 |

<!-- loc page=1 block=b_ff77d37f7efb type=table -->

SWIFT CSP assessment
<!-- loc page=1 block=b_f91d46179d47 type=text -->

![image](#b_8ca757279606)
<!-- loc page=1 block=b_8ca757279606 type=image -->

## Page 2

| Request for Proposal |  |
| --- | --- |
| Etex – SWIFT CSP assessment | Version: 1 |
| Date: 27 January 2026 | Page: 2 of 15 |

<!-- loc page=2 block=b_832da68ff19d type=table -->

### TABLE OF CONTENTS
<!-- loc page=2 block=b_0778de4e380e type=heading -->

Introduction............................................................................................................................... 3 1.1
<!-- loc page=2 block=b_94a202586aa7 type=text -->

Etex .................................................................................................................................. 3 1.2
<!-- loc page=2 block=b_b840730b09d6 type=text -->

RFP objectives and context ............................................................................................... 4 1.3
<!-- loc page=2 block=b_d3155676adcf type=text -->

Disclaimer ......................................................................................................................... 4 1.4
<!-- loc page=2 block=b_e5dd6cb70a87 type=text -->

Waiver .............................................................................................................................. 5 1.5
<!-- loc page=2 block=b_a433533935ba type=text -->

Confidentiality .................................................................................................................. 5 1.6
<!-- loc page=2 block=b_b17e040bde97 type=text -->

Definitions and Interpretations ......................................................................................... 5 Scope & TIMELINE ..................................................................................................................... 6 2.1
<!-- loc page=2 block=b_a2889e590209 type=text -->

High level Scope................................................................................................................ 6 2.2
<!-- loc page=2 block=b_a7322a30acf1 type=text -->

Etex Group (ETEXBEBB) ..................................................................................................... 6 2.3
<!-- loc page=2 block=b_839122558063 type=text -->

URSA Insulation (URSNESMMXXX) .................................................................................... 7 Timing ....................................................................................................................................... 8 General Assumptions: ................................................................................................................ 8 4.1.1
<!-- loc page=2 block=b_0afd12106a2e type=text -->

General requirements .................................................................................................. 8 4.2
<!-- loc page=2 block=b_da778108542c type=text -->

Carbon footprint and Corporate Social Responsibility ....................................................... 8 RFP Procedure ........................................................................................................................... 9 5.1
<!-- loc page=2 block=b_fac56f680041 type=text -->

Agenda and Timetable ...................................................................................................... 9 5.2
<!-- loc page=2 block=b_2849e3541b1d type=text -->

Acknowledgment .............................................................................................................. 9 5.3
<!-- loc page=2 block=b_f71e9c5f41c5 type=text -->

Submission of the RFP Response ................................................................................

…

## Loc comments

Same string written to the `.md` file / `document.to_dict()["markdown"]`. Filename belongs in YAML only, not in body text.

In [29]:
locs = [line for line in markdown.splitlines() if line.startswith("<!-- loc ")]
print(f"loc comments: {len(locs)}")
print("\n".join(locs[:12]))
print()
body = markdown.split("---", 2)[-1]
print("filename in body:", document.metadata.filename in body)

loc comments: 179
<!-- loc page=1 block=b_ff77d37f7efb type=table -->
<!-- loc page=1 block=b_f91d46179d47 type=text -->
<!-- loc page=1 block=b_8ca757279606 type=image -->
<!-- loc page=2 block=b_832da68ff19d type=table -->
<!-- loc page=2 block=b_0778de4e380e type=heading -->
<!-- loc page=2 block=b_94a202586aa7 type=text -->
<!-- loc page=2 block=b_b840730b09d6 type=text -->
<!-- loc page=2 block=b_d3155676adcf type=text -->
<!-- loc page=2 block=b_e5dd6cb70a87 type=text -->
<!-- loc page=2 block=b_a433533935ba type=text -->
<!-- loc page=2 block=b_b17e040bde97 type=text -->
<!-- loc page=2 block=b_a2889e590209 type=text -->

filename in body: False


## Save JSON

Writes `document.to_dict()` (the parse contract). Same payload as `test.py … parse`.

In [30]:
SAVE_JSON.parent.mkdir(parents=True, exist_ok=True)
SAVE_JSON.write_text(
    json.dumps(payload, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)
print(f"wrote {SAVE_JSON}  ({SAVE_JSON.stat().st_size:,} bytes)")

wrote D:\Work\Etex\ai_application\test_files\parser_test\sample_output.json  (167,899 bytes)
